In [1]:
%reload_ext autoreload
%autoreload 2

### Project Paths

In [2]:
from pathlib import Path
import sys
import importlib

# JAX must be configured before creating JAX arrays or importing project modules that create them.
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change this depending on notebook location:
# 0 = notebook is 1 folder inside project root, e.g. dp_termpaper/edu1
# 1 = notebook is 2 folders inside project root, e.g. dp_termpaper/data/moments
# 2 = notebook is 3 folders inside project root
amount_of_levels = 0

DIR = Path.cwd().resolve().parents[amount_of_levels]

if str(DIR) not in sys.path:
    sys.path.insert(0, str(DIR))

import project_paths as pp
from project_imports import *

importlib.reload(pp)

print("Project root:", pp.DIR)
print("JAX x64 enabled:", jax.config.read("jax_enable_x64"))

Project root: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\Thesis\dp_termpaper
JAX x64 enabled: True


## Load Data 

In [3]:
FILE_UDD1 = pp.MOMENTS_DIR / "moments_udd1.txt"
FILE_UDD2 = pp.MOMENTS_DIR / "moments_udd2.txt"
FILE_UDD3 = pp.MOMENTS_DIR / "moments_udd3.txt"

MORTALITY = pp.DATA_DIR / "mortality.xlsx"

In [ ]:
# For inspecting the dataset we read  all three moment datasets
udd1 = pd.read_csv(FILE_UDD1)
udd2 = pd.read_csv(FILE_UDD2)
udd3 = pd.read_csv(FILE_UDD3)

In [4]:
# Previously optimized parameters, kept for reference.
# The numerical convergence analysis below uses params_initial defined later.
PARAM_FILE = pp.STRUCTURAL_RESULTS_DIR / "optimized_params_udd1.txt"
params_old = load_params_txt(PARAM_FILE)
# params_old

In [5]:
# Read CSV
df_edu = pd.read_csv(FILE_UDD1)
# read mortality and discard seoncd and third column
df_mort = pd.read_excel(MORTALITY, sheet_name="DOD", usecols=[0, 3])

# 2) standardize colomn name and remove _FREQ_ column
# Iterate over DataFrames (if there are multiple DataFrames to process)
for data_frame in [df_edu]:
    # Rename ALDER → age
    if "ALDER" in data_frame.columns:
        data_frame.rename(columns={"ALDER": "age"}, inplace=True)
    # Remove _FREQ_-column if it exists
    if "_FREQ_" in data_frame.columns:
        data_frame.drop(columns=["_FREQ_"], inplace=True)
# drop var_wage, skew_wage and pens
df_edu.drop(columns=["var_wage", "skew_wage", "pens"], inplace=True)

# ====================================================================================================================

# Structural Estimation

### Structure
1. setup_model
2. validate_exogenous
3. solve model
4. simulate data
5. convert/inspect simulated data
6. compute simulated moments
7. compare simulated moments to empirical moments
8. only then structural estimation

## Initiatal parameters - some are estimated above, the rest to be structuraly estimated

In [7]:
beta0, beta1, beta2 = np.loadtxt(pp.FIRST_STAGE_RESULTS_DIR / "wage_params_udd1_ols.txt")
params_initial = {}
params_initial["interest_rate"] = 0.01
params_initial["income_shock_mean"] = 0.000 # income shock mean
params_initial["income_shock_std"] = 0.15721008127024574 # income shock scale
params_initial["taste_shock_scale"] = 1.0966242697158703 # taste shock scale

# discount factor
params_initial["discount_factor"] = 0.90535854

# parameters for the utility function
params_initial["rho"]=0.50092644 #0.76492381

# disutility of each hours choice
params_initial["gamma"]=jnp.array([0.82253966,1.5360604,1.93859858,1.25922689]) 

# wage parameters
params_initial["beta0"]=beta0
params_initial["beta1"]=beta1
params_initial["beta2"]=beta2

# Age component disutility
params_initial["kappa1"]=0.01531974
params_initial["kappa2"]=0.00216552

# transsition cost
params_initial["phi"]=0.94614399

# mortality parameters
params_initial["alpha1"]=0.000417
params_initial["alpha2"]=0.099277

# bequest parameters
params_initial["b_scale"]=1.2133177
params_initial["xi"]=0.26926199

# labor market parameters
params_initial["eta_edu"]=0.43848

params=params_initial.copy()

## Options for model - Choices and states

In [8]:
n_periods = 55
labour_choices = np.arange(5, dtype=np.int32)  # 5 choices
alpha1, alpha2 = np.loadtxt(pp.STRUCTURAL_RESULTS_DIR / "mortality_params.txt")

# model_config is generated below by make_model_config(...), so the numerical
# experiments can vary asset grid points, experience grid points, and quadrature points.

model_specs = {
    "n_periods": n_periods,
    "labour_choices": labour_choices,
    "hours": jnp.array([0, 250, 750, 1300, 1900], dtype=jnp.float64),
    "max_hours": 1500,
    "start_age": 30,
    "tax_threshold1": 0.480,
    "tax_threshold2": 5.698,
    "tax_base_rate": 0.38,
    "tax_top_rate": 0.5,
    "retirement_age": 67,
    "oap_base_amount": 0.80328,
    "oap_max_supplement": 0.92940,
    "supp_threshold": 0.79300,
    "oap_threshold": 3.3592,
    "supp_reduction_rate": 0.309,
    "oap_reduction_rate": 0.3,
    "alpha1": jnp.asarray(alpha1, dtype=jnp.float64),
    "alpha2": jnp.asarray(alpha2, dtype=jnp.float64),
    "max_init_experience": 5,
    "max_ret_period": 45,  # Age 75
    "min_ret_period": 30,  # Age 60
}

stochastic_states_transitions = {
    "survival": prob_survival,
}

hours_map = {
    i: int(h)
    for i, h in enumerate(np.asarray(model_specs["hours"]))
}

### ANALYSIS

In [9]:
def make_model_config(n_assets=20, n_experience=5, n_quad_points=5):
    return {
        "n_periods": n_periods,
        "choices": labour_choices,
        "n_quad_points": n_quad_points,
        "continuous_states": {
            "assets_end_of_period": jnp.linspace(
                0.0, 50.0, n_assets, dtype=jnp.float64
            ),
            "experience": jnp.linspace(
                0.0, 1.0, n_experience, dtype=jnp.float64
            ),
        },
        "stochastic_states": {
            "survival": [0, 1],
        },
    }


def make_initial_states(n_individuals, seed=132):
    key = jax.random.PRNGKey(seed)

    labels = jnp.array([0, 1, 2, 3, 4], dtype=jnp.int32)
    probs = jnp.array(
        [0.259155, 0.118310, 0.108099, 0.108451, 0.405986],
        dtype=jnp.float64,
    )

    lagged_choice = jax.random.choice(
        key,
        a=labels,
        shape=(n_individuals,),
        p=probs,
        replace=True,
    )

    states_initial = {
        "period": jnp.zeros(n_individuals, dtype=jnp.int32),
        "lagged_choice": lagged_choice,
        "experience": jnp.full(n_individuals, 1.0, dtype=jnp.float64),
        "survival": jnp.ones(n_individuals, dtype=jnp.int32),
        "assets_begin_of_period": jnp.full(
            n_individuals,
            0.505047,
            dtype=jnp.float64,
        ),
    }

    return states_initial


# Baseline configuration used for quick tests and as the default solver grid.
model_config = make_model_config(n_assets=20, n_experience=5, n_quad_points=5)

In [10]:
import time


POINT_MOMENT_EXCLUDE_SUFFIXES = ("_se", "_lower", "_upper")
POINT_MOMENT_EXCLUDE_COLUMNS = {"age", "period", "N", "n_individuals"}


def solve_model_for_config(label, n_assets, n_experience, n_quad_points):
    """Set up, validate, and solve the model for one numerical solver configuration."""
    model_config_exp = make_model_config(
        n_assets=n_assets,
        n_experience=n_experience,
        n_quad_points=n_quad_points,
    )

    model_exp = dcegm.setup_model(
        model_config=model_config_exp,
        model_specs=model_specs,
        utility_functions=utility_functions,
        utility_functions_final_period=final_period_utility,
        budget_constraint=budget_dcegm_initial,
        state_space_functions=create_state_space_function_dict(),
        stochastic_states_transitions=stochastic_states_transitions,
    )

    model_exp.validate_exogenous(params)

    t0 = time.time()
    model_solved_exp = model_exp.solve(params)
    solve_time = time.time() - t0

    return model_exp, model_solved_exp, solve_time


def simulate_from_solved_model(
    model_solved_exp,
    label,
    n_assets,
    n_experience,
    n_quad_points,
    n_individuals,
    solve_time=None,
    seed=132,
    store_sim=False,
):
    """Simulate from a fixed solved model and compute simulated moments with CIs."""
    states_initial = make_initial_states(
        n_individuals=n_individuals,
        seed=seed,
    )

    t0 = time.time()
    sim = model_solved_exp.simulate(
        states_initial=states_initial,
        seed=seed,
    )
    sim_time = time.time() - t0

    # Use the CI version because the plotting function expects *_lower and *_upper columns.
    moments = compute_simulation_moments_with_ci(
        sim,
        start_age=model_specs["start_age"],
        hours_map=hours_map,
    )

    out = {
        "label": label,
        "n_assets": n_assets,
        "n_experience": n_experience,
        "n_quad_points": n_quad_points,
        "n_individuals": n_individuals,
        "solve_time": solve_time,
        "sim_time": sim_time,
        "moments": moments,
    }

    # Keep this False for large runs unless you explicitly need the full simulated panel.
    if store_sim:
        out["sim"] = sim

    return out


def run_model_experiment(
    label,
    n_assets,
    n_experience,
    n_quad_points,
    n_individuals,
    seed=132,
    store_sim=False,
):
    """Convenience wrapper: solve one model configuration and simulate from it."""
    _, model_solved_exp, solve_time = solve_model_for_config(
        label=label,
        n_assets=n_assets,
        n_experience=n_experience,
        n_quad_points=n_quad_points,
    )

    return simulate_from_solved_model(
        model_solved_exp=model_solved_exp,
        label=label,
        n_assets=n_assets,
        n_experience=n_experience,
        n_quad_points=n_quad_points,
        n_individuals=n_individuals,
        solve_time=solve_time,
        seed=seed,
        store_sim=store_sim,
    )


def _moments_to_dataframe(moments):
    """Convert a moment object to a pandas DataFrame for comparison."""
    if isinstance(moments, pd.DataFrame):
        return moments.copy()
    if isinstance(moments, dict):
        return pd.DataFrame(moments)
    return pd.DataFrame(moments)


def get_numeric_moment_columns(moments, benchmark=None, exclude=POINT_MOMENT_EXCLUDE_COLUMNS):
    """
    Select point-estimate moment columns shared with the benchmark.

    CI columns are deliberately excluded from the convergence-distance metrics.
    """
    df = _moments_to_dataframe(moments)

    cols = []
    for col in df.columns:
        if col in exclude:
            continue
        if col.endswith(POINT_MOMENT_EXCLUDE_SUFFIXES):
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)

    if benchmark is not None:
        bench = _moments_to_dataframe(benchmark)
        cols = [col for col in cols if col in bench.columns]

    return cols


def moment_distance(moments, benchmark, moment_cols=None):
    """Compute distance between a result's point-estimate moments and benchmark moments."""
    df = _moments_to_dataframe(moments)
    bench = _moments_to_dataframe(benchmark)

    if "age" in df.columns and "age" in bench.columns:
        merged = df.merge(
            bench,
            on="age",
            suffixes=("", "_benchmark"),
            how="inner",
        )

        if moment_cols is None:
            moment_cols = get_numeric_moment_columns(df, bench)

        diff_cols = []
        for col in moment_cols:
            bench_col = f"{col}_benchmark"
            if col in merged.columns and bench_col in merged.columns:
                merged[f"{col}_diff"] = merged[col] - merged[bench_col]
                diff_cols.append(f"{col}_diff")

        diff = merged[diff_cols].to_numpy(dtype=float)
    else:
        if moment_cols is None:
            moment_cols = get_numeric_moment_columns(df, bench)

        m = df[moment_cols].to_numpy(dtype=float)
        b = bench[moment_cols].to_numpy(dtype=float)
        n_rows = min(m.shape[0], b.shape[0])
        diff = m[:n_rows] - b[:n_rows]

    return {
        "rmse": np.sqrt(np.nanmean(diff ** 2)),
        "mean_abs_diff": np.nanmean(np.abs(diff)),
        "max_abs_diff": np.nanmax(np.abs(diff)),
    }


def build_convergence_table(results, benchmark=None, benchmark_label="benchmark", moment_cols=None):
    """Create a paper-ready numerical convergence table from experiment results."""
    if benchmark is None:
        benchmark = results[-1]["moments"]

    rows = []

    for res in results:
        dist = moment_distance(
            moments=res["moments"],
            benchmark=benchmark,
            moment_cols=moment_cols,
        )

        rows.append({
            "label": res["label"],
            "n_assets": res["n_assets"],
            "n_experience": res["n_experience"],
            "n_quad_points": res["n_quad_points"],
            "n_individuals": res["n_individuals"],
            "solve_time": res["solve_time"],
            "sim_time": res["sim_time"],
            f"rmse_vs_{benchmark_label}": dist["rmse"],
            f"mean_abs_diff_vs_{benchmark_label}": dist["mean_abs_diff"],
            f"max_abs_diff_vs_{benchmark_label}": dist["max_abs_diff"],
        })

    return pd.DataFrame(rows)


# Determining simulation size

In [12]:
simulation_sizes = [10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]

# For simulation convergence, keep the solved policy functions fixed.
# This isolates Monte Carlo simulation noise from solver approximation error.
baseline_model, baseline_model_solved, baseline_solve_time = solve_model_for_config(
    label="baseline solver",
    n_assets=20,
    n_experience=5,
    n_quad_points=5,
)

simulation_results = []

for n_sim in simulation_sizes:
    result = simulate_from_solved_model(
        model_solved_exp=baseline_model_solved,
        label=f"N={n_sim}",
        n_assets=20,
        n_experience=5,
        n_quad_points=5,
        n_individuals=n_sim,
        solve_time=baseline_solve_time,
        seed=132,
        store_sim=False,
    )

    simulation_results.append(result)

    print(
        result["label"],
        "solve time:", result["solve_time"],
        "sim time:", result["sim_time"],
    )

# Use the largest simulation as the benchmark.
df_simulation_convergence = build_convergence_table(
    simulation_results,
    benchmark=simulation_results[-1]["moments"],
    benchmark_label=f"N_{simulation_sizes[-1]}",
)

display(df_simulation_convergence)

State specific choice set not provided. Assume all choices are available in every state.
Update function for state space not given. Assume states only change with an increase of the period and lagged choice.
Starting state space creation
State space created.

Starting state-choice space creation and child state mapping.
State, state-choice and child state mapping created.

Start creating batches for the model.
The batch size of the backwards induction is  25
Model setup complete.

N=10000 solve time: 3.3232269287109375 sim time: 2.588857650756836
N=25000 solve time: 3.3232269287109375 sim time: 3.939483404159546
N=50000 solve time: 3.3232269287109375 sim time: 6.0464112758636475
N=100000 solve time: 3.3232269287109375 sim time: 11.108913660049438
N=250000 solve time: 3.3232269287109375 sim time: 27.34465718269348
N=500000 solve time: 3.3232269287109375 sim time: 57.42675280570984
N=1000000 solve time: 3.3232269287109375 sim time: 137.65714573860168


,label,n_assets,n_experience,n_quad_points,n_individuals,solve_time,sim_time,rmse_vs_N_1000000,mean_abs_diff_vs_N_1000000,max_abs_diff_vs_N_1000000
0,N=10000,20,5,5,10000,3.323227,2.588858,4.893997,0.749736,100.084923
1,N=25000,20,5,5,25000,3.323227,3.939483,3.010162,0.483953,42.289408
2,N=50000,20,5,5,50000,3.323227,6.046411,2.539265,0.429626,35.052315
3,N=100000,20,5,5,100000,3.323227,11.108914,1.573117,0.243326,27.187589
4,N=250000,20,5,5,250000,3.323227,27.344657,0.930228,0.161464,12.892584
5,N=500000,20,5,5,500000,3.323227,57.426753,0.713187,0.122762,12.363243
6,N=1000000,20,5,5,1000000,3.323227,137.657146,0.000000,0.000000,0.000000


In [ ]:
solver_configs = [
    {
        "label": "baseline",
        "n_assets": 20,
        "n_experience": 5,
        "n_quad_points": 5,
    },
    {
        "label": "medium",
        "n_assets": 50,
        "n_experience": 10,
        "n_quad_points": 7,
    },
    {
        "label": "high",
        "n_assets": 100,
        "n_experience": 15,
        "n_quad_points": 9,
    },
]

solver_results = []

for cfg in solver_configs:
    result = run_model_experiment(
        label=cfg["label"],
        n_assets=cfg["n_assets"],
        n_experience=cfg["n_experience"],
        n_quad_points=cfg["n_quad_points"],
        n_individuals=100_000,
        seed=132,
        store_sim=False,
    )

    solver_results.append(result)

    print(
        result["label"],
        "solve time:", result["solve_time"],
        "sim time:", result["sim_time"],
    )

# Use the high-accuracy solver as the benchmark.
df_solver_convergence = build_convergence_table(
    solver_results,
    benchmark=solver_results[-1]["moments"],
    benchmark_label="high",
)

display(df_solver_convergence)

State specific choice set not provided. Assume all choices are available in every state.
Update function for state space not given. Assume states only change with an increase of the period and lagged choice.
Starting state space creation
State space created.

Starting state-choice space creation and child state mapping.
State, state-choice and child state mapping created.

Start creating batches for the model.
The batch size of the backwards induction is  25
Model setup complete.

baseline solve time: 1.7397267818450928 sim time: 7.276346683502197
State specific choice set not provided. Assume all choices are available in every state.
Update function for state space not given. Assume states only change with an increase of the period and lagged choice.
Starting state space creation
State space created.

Starting state-choice space creation and child state mapping.
State, state-choice and child state mapping created.

Start creating batches for the model.
The batch size of the backwards 

,label,n_assets,n_experience,n_quad_points,n_individuals,solve_time,sim_time,rmse_vs_high,mean_abs_diff_vs_high,max_abs_diff_vs_high
0,baseline,20,5,5,100000,1.739727,7.276347,9.322539,2.159531,107.728730
1,medium,50,10,7,100000,2.503918,14.084445,9.012284,1.089871,135.151831
2,high,100,15,9,100000,2.266991,28.365450,0.000000,0.000000,0.000000


## Moments and plots

Use the simulated moments stored in `simulation_results` and `solver_results`. The full simulated panel is not kept in memory by default.


In [ ]:
labels = {
    "avg_wealth": "Average Wealth",
    "hours_0": "Unemployed",
    "hours_1": "Below 10 hours",
    "hours_2": "10-20 hours",
    "hours_3": "20-30 hours",
    "hours_4": "Above 30 hours",
    "avg_hours": "Average Hours",
    "prob_work": "Employment Rate",
    "avg_wage": "Average Wage",
    "avg_experience": "Average Experience",
    "work_work": "Work to Work Transition Rate",
    "nowork_nowork": "No Work to No Work Transition Rate",
}

scales = {
    "avg_wealth": 100_000,
    "avg_wage": 100_000,
}

ylims = {
    "avg_wealth": (0, 3_000_000),
    "avg_hours": (0, 2_000),
    "prob_work": (0, 1),
    "hours_0": (0, 1),
    "hours_1": (0, 1),
    "hours_2": (0, 1),
    "hours_3": (0, 1),
    "hours_4": (0, 1),
    "avg_wage": (0, 500),
}


def empirical_moments_for_plot(max_age=75):
    """Prepare empirical moments for the plotting function."""
    edu = df_edu.copy()

    if "ALDER" in edu.columns and "age" not in edu.columns:
        edu = edu.rename(columns={"ALDER": "age"})

    return edu.loc[edu["age"] <= max_age].copy()


def simulated_moments_for_plot(result, max_age=75):
    """Extract simulated moments from one experiment result and filter to the plotted age range."""
    moments_sim = result["moments"].copy()

    if "age" not in moments_sim.columns:
        raise KeyError(
            "The simulated moments do not contain an 'age' column. "
            "Rerun the experiment cells after updating compute_simulation_moments_with_ci."
        )

    required_ci_cols = ["prob_work_lower", "prob_work_upper"]
    missing_ci_cols = [col for col in required_ci_cols if col not in moments_sim.columns]
    if missing_ci_cols:
        raise KeyError(
            f"Missing CI columns: {missing_ci_cols}. "
            "Rerun the simulation/solver experiment cells so moments are computed with "
            "compute_simulation_moments_with_ci(...)."
        )

    return moments_sim.loc[moments_sim["age"] <= max_age].copy()


def plot_result_against_empirical(result, out_subfolder, max_age=75):
    """Plot one stored experiment result against empirical moments."""
    edu = empirical_moments_for_plot(max_age=max_age)
    moments_sim = simulated_moments_for_plot(result, max_age=max_age)

    plot_empirical_vs_simulated_with_ci(
        edu=edu,
        moments_sim=moments_sim,
        out_base_dir=pp.SIM_PLOTS_DIR,
        out_subfolder=out_subfolder,
        var_labels=labels,
        var_scales=scales,
        ylims=ylims,
    )

    return moments_sim
